# Extracting structured data from PDFs — what we are building today

Before we take anything apart, let's watch it work.

Below, we point one function at a folder of PDFs, tell it what we want, and get
a table back. The folder holds two taxonomic papers: one born-digital from
2013, one a 1929 scan that underwent OCR years ago with outdated tools (and so the text has multiple errores).

The program will use AI reasoning to identify whether a PDF can be taken at face value, or whether we need to use another AI model to redo the OCR. Either way, it will extract text and figures automatically. Then it will proceed to extract structured data that we requested with our **prompt** and **schema**, and show us a data table.

For now this is a black box, don't worry. Let's just run, see the result, and then we will spend the rest of the day working on the details. After the workshop, you will be able to adapt the code in pdf_extraction.py for your own use.


**Set your runtime to a GPU first:** Runtime → Change runtime type → T4. Then
click *Connect*.

## Setup

This installs Ollama and downloads two models, which is about 14 GB. It will take a few minutes, and we have time to talk after.

In [1]:
import glob, os, subprocess, sys, time

IN_COLAB = "google.colab" in sys.modules
REPO = "2026_UIUC_workshop_llm_pdf_extraction"
BRANCH = "main"   # switch to your working branch to test unmerged changes

if IN_COLAB:
    # zstd unpacks Ollama's installer; pciutils lets it find the GPU.
    !DEBIAN_FRONTEND=noninteractive apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip -q install ollama pymupdf pandas pillow
    if os.path.basename(os.getcwd()) != REPO:   # so this cell is safe to re-run
        if not os.path.isdir(REPO):
            !git clone -q --branch {BRANCH} https://github.com/de-Medeiros-insect-lab/{REPO}.git
        os.chdir(REPO)
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Jupyter puts the notebook's own folder on the import path, not the folder we
# are working in, so say explicitly where pdf_extraction.py lives.
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

if not os.path.exists("pdf_extraction.py"):
    raise RuntimeError(
        f"pdf_extraction.py is not here. We are in {os.getcwd()}, checked out "
        f"from branch {BRANCH!r} of {REPO} -- is that the branch the file is on?")

import ollama

def server_ready(timeout=120):
    """Wait until Ollama answers, or give up."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            ollama.list()
            return True
        except Exception:
            time.sleep(1)
    return False

assert server_ready(), "Ollama did not start"
print("Ollama is up")

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama is up


In [2]:
# ~6.5 GB: reads text and images, and follows a schema.
!ollama pull qwen3.5:9b
# ~6.7 GB: transcribes a page image. Only used on pages that need it.
!ollama pull deepseek-ocr

## Input

We need three things:

- **a folder** of PDFs,
- **a schema** — the fields you want, and their types,
- **a prompt** — what to extract, and what each field means.

The result will be one table.

In [5]:
from pdf_extraction import extract_folder

SCHEMA = {
    "type": "object",
    "properties": {
        "species": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name":       {"type": "string"},
                    "author":     {"type": "string"},
                    "min_length": {"type": "number"},
                    "max_length": {"type": "number"},
                    "localities": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "country": {"type": "string"},
                                "precise_locality": {"type": "string"}
                            },
                            "required": ["country", "precise_locality"],
                        }
                    }
                },
                "required": ["name"],
                "additionalProperties": False,
            }
        }
    },
    "required": ["species"],
    "additionalProperties": False,
}

PROMPT = (
    '''List every species this paper describes.
    'name' is the species name
    'author' is the taxonomic authority for that name -- the person who described it -- not necessarily an author of this paper.
    'min_length' and 'max_length' are the smallest and largest body length in mm given for the species.
    'localities' is an array with each locality where the species was collected.
    Do not invent values that are not stated.

    '''
)

df = extract_folder("example_pdfs", prompt=PROMPT, schema=SCHEMA)
df

deciding how to read each document
  Marshall1929_AnnMagNatHist.pdf: re-reading every page — Set needs_ocr to true because the text contains numerous OCR artifacts and impossible spellings such as "Congi~odus leucopw~cilus" and "Fi.q.".
  deMedeiros2013Zootaxa.pdf: using the stored text — needs_ocr is false because the text reads cleanly with correct scientific spelling such as 'Anchylorhynchus centrosquamatus' and proper punctuation like '15/VII/2008', indicating it is stored directly in the PDF rather than being OCR output.
processing Marshall1929_AnnMagNatHist.pdf
    page 1: already done
    page 2: already done
    page 3: already done
    page 4: already done
    page 5: already done
    page 6: already done
    page 7: already done
    page 8: already done
processing deMedeiros2013Zootaxa.pdf
    page 1: already done
    page 2: already done
    page 3: already done
    page 4: already done
    page 5: already done
    page 6: already done
    page 7: already done
extracting fro

,source,name,author,min_length,max_length,localities
0,Marshall1929_AnnMagNatHist.pdf,Phelypera pachire,Marshall,6.6,8.4,[]
1,Marshall1929_AnnMagNatHist.pdf,Conotrachelus myricariae,Marshall,6.0,6.3,[]
2,Marshall1929_AnnMagNatHist.pdf,Huarucus cacti,Marshall,4.2,6.0,[]
3,Marshall1929_AnnMagNatHist.pdf,Microtrates ypsilon,Marshall,3.3,4.8,[]
4,deMedeiros2013Zootaxa.pdf,Anchylorhynchus pinocchio,"De Medeiros & Núñez-Avellaneda, 2013",4.5,5.7,"[{'country': 'Colombia', 'precise_locality': '..."
5,deMedeiros2013Zootaxa.pdf,Anchylorhynchus centrosquamatus,"De Medeiros & Núñez-Avellaneda, 2013",4.7,5.9,"[{'country': 'Colombia', 'precise_locality': '..."
6,deMedeiros2013Zootaxa.pdf,Anchylorhynchus luteobrunneus,"De Medeiros & Núñez-Avellaneda, 2013",3.9,4.8,"[{'country': 'Colombia', 'precise_locality': '..."


## What just happened

Look at the progress lines. Before reading either paper, the function asked a
reasoning model one question about it — *can this document's own text be
trusted?* — and then printed what it decided and why.

It redid the OCR when needed and found the figures in both papers. Then it asked for the
fields in `SCHEMA` and made the request in `PROMPT`.

This took some time because we are starting everything from scratch and working with a somewhat limited hardware. But there are multiple options to speed up or make it more accurate.

**Now check the table against the papers.** Does it look right?

## Your own PDFs

If you ever want to run this exercise on your own pdfs, you can adapt the code below, which reads PDFs from a folder called `my_pdfs`. Upload a few PDFs with the folder icon 📁 in the
left sidebar, then edit the schema and prompt for what *you* want out of them.

Start with two or three documents. Test prompts and schemas.

Two arguments worth knowing about when you come back to this with real work:

- `needs_ocr=True` or `needs_ocr=False` skips the AI reasoning when you know alrady if PDFs need OCR or not.
- `out_dir="somewhere"` chooses where the processed folders are written
  (`processed/` by default). Open them: one markdown file per page, in
  reading order, with the figures linked where they appeared. Pages
  already there are never redone, so an interrupted run resumes, and a
  page you delete is the only one re-read.

In [4]:
os.makedirs("my_pdfs", exist_ok=True)

MY_SCHEMA = {
    "type": "object",
    "properties": {
        "records": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "FIELD_ONE": {"type": "string"},
                    "FIELD_TWO": {"type": "string"},
                },
                "required": ["FIELD_ONE"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["records"],
    "additionalProperties": False,
}

MY_PROMPT = (
    "DESCRIBE WHAT TO EXTRACT HERE. FIELD_ONE is ... FIELD_TWO is ... "
    "Do not invent values that are not stated.\n\n"
)

if not glob.glob("my_pdfs/*.pdf"):
    print("Upload PDFs into my_pdfs/ with the folder icon in the sidebar, "
          "then run this cell again.")
else:
    my_df = extract_folder("my_pdfs", prompt=MY_PROMPT, schema=MY_SCHEMA)
    my_df.to_csv("my_results.csv", index=False)
    display(my_df)

Upload PDFs into my_pdfs/ with the folder icon in the sidebar, then run this cell again.


## Now let's take it apart

Everything you just ran lives in [`pdf_extraction.py`](pdf_extraction.py) — a
few hundred lines, yours to keep, reuse, rewrite and to point at your own folders. You are
welcome to read it now, but it will not teach you much on its own: the
interesting parts are the decisions, not the code.

So now we will **close this notebook, disconnect the runtime, and open `workshop.ipynb`.**
We will start from an empty cell and build up to what you just saw, in five
sessions:

1. **Setup** — talking to a language model from Python
2. **Preparing documents** — OCR and figure extraction
3. **Structured extraction** — getting tables out of unstructured text
4. **An agent** — letting the model choose its own approach
5. **Scaling up** — a bigger model in the cloud, and a whole folder of PDFs